[◀ Notebook 17: Modern Packaging](17_modern_packaging.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 19: Memory Management ▶](19_memory_management.ipynb)

# Notebook 18: Testing, Mocking & Code Quality

Welcome to Notebook 18 of the Bottom-Up Python Tutorial. In this notebook, we focus on writing robust, clean, and testable code. We will examine unit testing using `pytest`, learn how to mock dependencies with `unittest.mock`, and explore static analysis and formatting using the Rust-based tool `ruff`.

### References:
- [Pytest Documentation](https://docs.pytest.org/)
- [Python standard library: unittest.mock](https://docs.python.org/3/library/unittest.mock.html)
- [Ruff Documentation](https://docs.astral.sh/ruff/)

## 1. Unit Testing with `pytest`

Testing is essential for building maintainable code. The `pytest` framework is the industry standard for Python:
- **Simpler Assertions:** You write standard Python `assert expression` statements instead of `self.assertEqual(...)`.
- **Test Discovery:** Pytest automatically finds and runs files named `test_*.py` or `*_test.py` containing functions prefixed with `test_`.
- **Parameterization:** You can run a single test function with multiple sets of arguments using the `@pytest.mark.parametrize` decorator.

### Pytest Fixtures
Fixtures are functions that set up a state or provide dependencies to tests. They use dependency injection—a test function requests a fixture by including its name in its argument list.
Fixtures can have different lifetimes (scopes):
- `function` (default): Setup/teardown runs before and after each test.
- `module`: Setup/teardown runs once per test file.
- `session`: Setup/teardown runs once for the entire test suite.

In [ ]:
import pytest

# Simple example of a function to test
def calculate_total(prices, tax_rate):
    if tax_rate < 0:
        raise ValueError("Tax rate cannot be negative")
    return sum(prices) * (1 + tax_rate)

# Example of a test using parameterization
@pytest.mark.parametrize("prices, tax_rate, expected", [
    ([10, 20], 0.1, 33.0),
    ([5, 5, 5], 0.0, 15.0),
    ([], 0.05, 0.0),
])
def test_calculate_total(prices, tax_rate, expected):
    assert calculate_total(prices, tax_rate) == pytest.approx(expected)

def test_calculate_total_invalid_tax():
    with pytest.raises(ValueError, match="Tax rate cannot be negative"):
        calculate_total([10, 20], -0.1)

## 2. Mocking with `unittest.mock`

Unit tests should test components in isolation. If a class depends on a database connection or a network client, we do not want our test to make real network requests or write to a database. Instead, we swap the dependencies with **Mock** objects.

### Core Concepts:
- **`Mock`:** An object that intercepts calls and returns other mock objects. You can set return values (`mock.return_value = 42`) or make it raise exceptions (`mock.side_effect = ConnectionError`).
- **`MagicMock`:** A subclass of `Mock` that implements Python's magic methods (e.g., `__len__`, `__getitem__`, `__str__`). Used for container emulation or complex objects.
- **`patch`:** A context manager or decorator that temporarily replaces target objects or modules (often global library targets like `requests.get`) with a mock object during a test.

In [ ]:
from unittest.mock import Mock, patch

class UserFetcher:
    def __init__(self, api_client):
        self.api_client = api_client

    def get_user_name(self, user_id):
        response = self.api_client.get(f"/users/{user_id}")
        return response.get("name", "Unknown")

def test_user_fetcher_success():
    # Create a mock API client
    mock_client = Mock()
    # Set its get method to return a specific dictionary
    mock_client.get.return_value = {"name": "Alice", "id": 1}
    
    fetcher = UserFetcher(mock_client)
    name = fetcher.get_user_name(1)
    
    assert name == "Alice"
    # Verify that get() was called exactly once with the correct URL
    mock_client.get.assert_called_once_with("/users/1")

test_user_fetcher_success()
print("Mock test completed successfully!")

## 3. Code Quality: Linters & Formatters (Ruff)

- **Linters:** Inspect code statically without executing it, flagging syntax errors, bugs, or stylistic violations (e.g., unused imports, undefined variables, complexity limits).
- **Formatters:** Automatically rewrite code files to conform to style guides (like PEP 8).
- **Ruff:** A modern, unified Python linter and formatter written in Rust. It compiles hundreds of times faster than traditional tools like `flake8` or `black` and provides identical diagnostics.

---

## 4. Coding Challenges

Complete the two challenges below.

### Challenge 1: Testing an Unstable Network Endpoint via Mocks

We have a class `WeatherFetcher` that depends on a `RequestsClient` to make web requests. 
If the network request succeeds, it returns the temperature. If it fails (HTTP status not 200) or throws an exception (like timeout/network down), the method should catch it and return a default value of `-999.0`.

Write a test suite in a file `assets/test_weather.py` using `pytest` and `unittest.mock` to test these scenarios:
1. **`test_fetch_temperature_success`**: Mock `client.get` to return a response with `status_code = 200` and `json()` returning `{"temp": 22.5}`. Assert the fetcher returns `22.5`.
2. **`test_fetch_temperature_http_error`**: Mock `client.get` to return `status_code = 500`. Assert the fetcher returns `-999.0`.
3. **`test_fetch_temperature_network_exception`**: Mock `client.get` to raise `ConnectionError`. Assert the fetcher returns `-999.0`.

The code structure for the classes is provided below. You must write the test functions into `assets/test_weather.py` and run the verification test.

In [ ]:
import os

# Create the application code file
app_code = """
class RequestsClient:
    def get(self, url):
        raise NotImplementedError("Real network calls are disabled in tests!")

class WeatherFetcher:
    def __init__(self, client):
        self.client = client

    def fetch_temperature(self, city):
        try:
            response = self.client.get(f"https://api.weather.com/v1/{city}")
            if response.status_code == 200:
                return response.json().get("temp")
            return -999.0
        except Exception:
            return -999.0
"""

with open("assets/weather_app.py", "w") as f:
    f.write(app_code)

# TODO: Write the test suite code as a string and save it to `assets/test_weather.py`.
# Import Mock from unittest.mock and classes from weather_app.
test_suite_code = """
from unittest.mock import Mock
from weather_app import WeatherFetcher, RequestsClient
import pytest

# Define your 3 test functions here
"""

with open("assets/test_weather.py", "w") as f:
    f.write(test_suite_code)

In [ ]:
# Solution Code for Challenge 1
# test_suite_code = """
# from unittest.mock import Mock
# from weather_app import WeatherFetcher
# 
# def test_fetch_temperature_success():
#     mock_client = Mock()
#     mock_response = Mock()
#     mock_response.status_code = 200
#     mock_response.json.return_value = {"temp": 22.5}
#     mock_client.get.return_value = mock_response
#     
#     fetcher = WeatherFetcher(mock_client)
#     assert fetcher.fetch_temperature("London") == 22.5
#     mock_client.get.assert_called_once_with("https://api.weather.com/v1/London")
# 
# def test_fetch_temperature_http_error():
#     mock_client = Mock()
#     mock_response = Mock()
#     mock_response.status_code = 500
#     mock_client.get.return_value = mock_response
#     
#     fetcher = WeatherFetcher(mock_client)
#     assert fetcher.fetch_temperature("London") == -999.0
# 
# def test_fetch_temperature_network_exception():
#     mock_client = Mock()
#     mock_client.get.side_effect = ConnectionError("Network down!")
#     
#     fetcher = WeatherFetcher(mock_client)
#     assert fetcher.fetch_temperature("London") == -999.0
# """
# with open("assets/test_weather.py", "w") as f:
#     f.write(test_suite_code)

In [ ]:
# Challenge 1 Verification Test
# We will run pytest on `assets/test_weather.py` programmatically
import subprocess
import sys

# Install pytest if not present
subprocess.run([sys.executable, "-m", "pip", "install", "pytest"], capture_output=True)

# Run pytest
result = subprocess.run(
    [sys.executable, "-m", "pytest", "assets/test_weather.py", "-v"],
    capture_output=True,
    text=True
)
print(result.stdout)
print(result.stderr)

assert result.returncode == 0, "Pytest run failed! Check your test assertions and mocked behavior."
print("Challenge 1: Success! All unit tests passed.")

### Challenge 2: Ruff Lint and Fix

We have generated a messy code file `assets/messy_script.py` which contains several PEP 8 and Python lint errors. 
Your challenge is to run `ruff check --fix` and `ruff format` using Python's subprocess tools (or manually editing) to fix all issues. 
The test asserts that running `ruff check assets/messy_script.py` returns no errors.

In [ ]:
# Set up the messy script file
messy_code = """
import sys
import os  # Unused import

def Add(a, b):
    return a+b

x  =  Add( 10,20 )
"""

with open("assets/messy_script.py", "w") as f:
    f.write(messy_code)

In [ ]:
# TODO: Clean up assets/messy_script.py so it conforms to Ruff lint rules. 
# You can run subprocess commands like:
# subprocess.run([sys.executable, "-m", "pip", "install", "ruff"])
# subprocess.run([sys.executable, "-m", "ruff", "check", "--fix", "assets/messy_script.py"])
# subprocess.run([sys.executable, "-m", "ruff", "format", "assets/messy_script.py"])
# 
# Or you can overwrite 'assets/messy_script.py' manually with clean code if you prefer.

In [ ]:
# Solution Code for Challenge 2
# import subprocess
# import sys
# subprocess.run([sys.executable, "-m", "pip", "install", "ruff"], capture_output=True)
# subprocess.run([sys.executable, "-m", "ruff", "check", "--fix", "assets/messy_script.py"], capture_output=True)
# subprocess.run([sys.executable, "-m", "ruff", "format", "assets/messy_script.py"], capture_output=True)

In [ ]:
# Challenge 2 Verification Test
import subprocess
import sys

# Ensure ruff is installed
subprocess.run([sys.executable, "-m", "pip", "install", "ruff"], capture_output=True)

# Run ruff check on assets/messy_script.py
result = subprocess.run(
    [sys.executable, "-m", "ruff", "check", "assets/messy_script.py"],
    capture_output=True,
    text=True
)
print("Ruff Output:\n", result.stdout)
print("Ruff Exit Code:", result.returncode)

assert result.returncode == 0, f"Ruff check found issues! Output: {result.stdout}"
print("Challenge 2: Success! Messy script is now clean and compliant.")

[◀ Notebook 17: Modern Packaging](17_modern_packaging.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 19: Memory Management ▶](19_memory_management.ipynb)